In [1]:
print("GPT-2 End-to-End: CLM (causal language modeling) Fine-tuning, Hyperparameter Search, Checkpoint Resume, Generation 🚀")

GPT-2 End-to-End: CLM (causal language modeling) Fine-tuning, Hyperparameter Search, Checkpoint Resume, Generation 🚀


In [2]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    TextStreamer,
    TextIteratorStreamer,
    EarlyStoppingCallback,
)
from datasets import load_dataset
from tqdm import tqdm
import numpy as np
import torch
import math
import threading

# ── TRAINER — CAUSAL LANGUAGE MODELING (GPT-2 on WikiText-2) ─────────────────
# Task         : Next-token prediction — given tokens 0..t, predict token t+1
#                Input  : raw English text, concatenated and chunked into fixed
#                         blocks (not padded per-example like classification)
#                Output : logits over the 50,257-token vocab at every position
#
# Architecture : GPT-2 (small, 124M)
#                Backbone : 12 decoder-only transformer blocks, hidden dim 768
#                           causal (masked) self-attention — token t can only
#                           attend to tokens 0..t, never future tokens
#                           BPE tokenizer — vocab size 50,257
#                Head     : Linear(768 -> 50257), tied with the input embedding
#                           weight matrix (weight tying — same matrix used for
#                           both the embedding lookup and the output projection)
#                Loss     : CrossEntropyLoss over shifted labels
#                           labels = input_ids shifted left by 1 internally
#
# Training     : Full fine-tuning (all 124M params), not LoRA
#                Optimizer : AdamW
#                Schedule  : linear warmup -> linear decay
#                Evaluation: perplexity = exp(eval_loss) after every epoch
#
# Data         : WikiText-2-raw — ~2M training tokens, Wikipedia articles
#                text is concatenated across examples then split into fixed
#                block_size chunks — CLM training does NOT pad per-example
#
# Key difference from the IMDb classification notebook:
#   - No [CLS] token, no sentence-level head. Every position produces a
#     prediction, and every position contributes to the loss.
#   - DataCollatorForLanguageModeling replaces DataCollatorWithPadding.
#   - GPT-2 has NO pad token by default — must be set explicitly or the
#     tokenizer throws on batched encoding.
# ─────────────────────────────────────────────────────────────────────────────

In [3]:
# ── DEVICE CHECK ──────────────────────────────────────────────────────────────
print(f"  CUDA available : {torch.cuda.is_available()}")
print(f"  GPU name       : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"  GPU memory     : {torch.cuda.get_device_properties(0).total_memory // 1024**3} GB" if torch.cuda.is_available() else "  GPU memory     : N/A")
# Expected on a Colab T4:
#   CUDA available : True
#   GPU name       : Tesla T4
#   GPU memory     : 14 GB

  CUDA available : True
  GPU name       : Tesla T4
  GPU memory     : 14 GB


In [4]:
# ── STEP 1 : Load dataset ─────────────────────────────────────────────────────
# WikiText-2-raw — small enough to fine-tune in a single Colab session
# dataset["train"] ~36,718 lines, dataset["validation"] ~3,760 lines
# Each row: {"text": "..."}  — many rows are empty strings or section headers,
# which is normal for WikiText's raw format (they get merged in Step 3)

dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

print("-- Dataset loaded --")
print(f"  Train rows      : {dataset['train'].num_rows}")
print(f"  Validation rows : {dataset['validation'].num_rows}")
print(f"  Sample row      : {repr(dataset['train'][5]['text'][:80])}")
# e.g. " The game began development in 2010 , carrying over a large portion of ..."

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

-- Dataset loaded --
  Train rows      : 36718
  Validation rows : 3760
  Sample row      : ' It met with positive sales in Japan , and was praised by both Japanese and west'


In [5]:
# ── STEP 2 : Load tokenizer ───────────────────────────────────────────────────
# GPT-2 uses byte-pair encoding (BPE), not WordPiece
# Critically: GPT-2 was NOT pretrained with a pad token, because CLM training
# never needed one (blocks are pre-chunked to equal length in Step 3).
# We still set pad_token = eos_token so batched generation later doesn't crash.

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

print("\n-- Tokenizer loaded --")
print(f"  Tokenizer class : {type(tokenizer).__name__}")
#   GPT2TokenizerFast
print(f"  Vocab size      : {tokenizer.vocab_size}")
#   50257
print(f"  [EOS] token id  : {tokenizer.eos_token_id}")
#   50256
print(f"  [PAD] token id  : {tokenizer.pad_token_id}  (aliased to EOS, GPT-2 has no native pad token)")
#   50256

sample_enc = tokenizer("Rohit trained the model overnight.", return_tensors="pt")
print(f"  Sample input_ids : {sample_enc['input_ids']}")


-- Tokenizer loaded --
  Tokenizer class : GPT2Tokenizer
  Vocab size      : 50257
  [EOS] token id  : 50256
  [PAD] token id  : 50256  (aliased to EOS, GPT-2 has no native pad token)
  Sample input_ids : tensor([[   49,  1219,   270,  8776,   262,  2746, 13417,    13]])


In [6]:
# ── STEP 3 : Tokenize and group into fixed-length blocks ─────────────────────
# CLM training does NOT pad each example individually. Instead:
#   1. Tokenize every row (no truncation, no padding)
#   2. Concatenate ALL token ids across the entire split into one long stream
#   3. Chop that stream into contiguous chunks of block_size tokens
#
# Why concatenate first? Individual WikiText rows vary from 0 tokens (blank
# lines) to a few hundred. Padding each one to block_size would waste massive
# compute on empty rows. Concatenating uses every token productively.
#
# block_size=128 keeps this runnable on a single T4; GPT-2's real context
# window is 1024, use that if you have more GPU memory.

block_size = 128

def tokenize_fn(batch):
    return tokenizer(batch["text"])

def group_texts(batch):
    # batch is a dict of lists after tokenize_fn + batched map, e.g.
    #   {"input_ids": [[...], [...], ...], "attention_mask": [[...], ...]}
    concatenated = {k: sum(batch[k], []) for k in batch.keys()}
    total_len = len(concatenated["input_ids"])
    total_len = (total_len // block_size) * block_size   # drop the remainder
    result = {
        k: [t[i : i + block_size] for i in range(0, total_len, block_size)]
        for k, t in concatenated.items()
    }
    # CLM labels are identical to input_ids — Trainer/model shifts internally
    result["labels"] = result["input_ids"].copy()
    return result

print("\n-- Tokenizing --")
tokenized = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing",
)

print("-- Grouping into blocks --")
lm_dataset = tokenized.map(
    group_texts,
    batched=True,
    batch_size=1000,
    desc="Grouping",
)

print(f"  Train blocks      : {lm_dataset['train'].num_rows}")
print(f"  Validation blocks : {lm_dataset['validation'].num_rows}")
print(f"  Block length      : {len(lm_dataset['train'][0]['input_ids'])}")
#   128, every block identical length -- no padding needed at all
print(f"  Columns           : {lm_dataset['train'].column_names}")
#   ['input_ids', 'attention_mask', 'labels']


-- Tokenizing --


Tokenizing:   0%|          | 0/4358 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/36718 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/3760 [00:00<?, ? examples/s]

-- Grouping into blocks --


Grouping:   0%|          | 0/4358 [00:00<?, ? examples/s]

Grouping:   0%|          | 0/36718 [00:00<?, ? examples/s]

Grouping:   0%|          | 0/3760 [00:00<?, ? examples/s]

  Train blocks      : 18667
  Validation blocks : 1930
  Block length      : 128
  Columns           : ['input_ids', 'attention_mask', 'labels']


In [7]:
# ── STEP 4 : Data collator ────────────────────────────────────────────────────
# DataCollatorForLanguageModeling(mlm=False) for CLM:
#   - stacks the pre-chunked blocks into a batch tensor (no padding needed,
#     every block is already exactly block_size long from Step 3)
#   - internally shifts labels by one position so position t predicts t+1
#   - mlm=False means "do NOT randomly mask tokens" (that's BERT's job) --
#     every token in every block is used as a training signal here

collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

print("\n-- Data collator created --")
print(f"  Collator type : {type(collator).__name__}")
#   DataCollatorForLanguageModeling
print(f"  mlm            : {collator.mlm}")
#   False -> causal LM mode

manual_batch = [lm_dataset["train"][i] for i in range(4)]
collated = collator(manual_batch)
print(f"  Batch input_ids shape : {collated['input_ids'].shape}")
#   torch.Size([4, 128])
print(f"  Batch labels shape    : {collated['labels'].shape}")
#   torch.Size([4, 128])


-- Data collator created --
  Collator type : DataCollatorForLanguageModeling
  mlm            : False
  Batch input_ids shape : torch.Size([4, 128])
  Batch labels shape    : torch.Size([4, 128])


In [8]:
# ── STEP 5 : model_init (NOT a fixed model) ───────────────────────────────────
# We define model_init() instead of loading `model` directly, because
# Step 8 (hyperparameter_search) needs a fresh, untrained model for every
# trial -- reusing one `model` object would carry over weights between trials
# and invalidate the search.
#
# GPT-2 must also have its embedding matrix resized IF the tokenizer's vocab
# changed (it hasn't here -- pad_token was aliased to an existing id, not a
# new one -- so resize_token_embeddings is a no-op safety call, not strictly
# required in this specific case).

def model_init():
    model = AutoModelForCausalLM.from_pretrained("gpt2")
    model.resize_token_embeddings(len(tokenizer))
    return model

probe_model = model_init()
total_params = sum(p.numel() for p in probe_model.parameters())
print("\n-- model_init defined --")
print(f"  Model class      : {type(probe_model).__name__}")
#   GPT2LMHeadModel
print(f"  Hidden size      : {probe_model.config.n_embd}")
#   768
print(f"  Num layers       : {probe_model.config.n_layer}")
#   12
print(f"  Total parameters : {total_params:,}")
#   124,439,808  (~124M)
del probe_model

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


-- model_init defined --
  Model class      : GPT2LMHeadModel
  Hidden size      : 768
  Num layers       : 12
  Total parameters : 124,439,808


In [9]:
# ── STEP 6 : compute_metrics — perplexity ─────────────────────────────────────
# CLM doesn't have "accuracy" in the classification sense -- the standard
# metric is perplexity = exp(cross_entropy_loss).
# Interpretation: perplexity is roughly "how many tokens the model is
# effectively choosing between, on average, at each position." Lower is
# better. A well fine-tuned small GPT-2 on in-domain text typically lands
# somewhere in the 20-40 range; the pretrained-but-unfinetuned baseline on
# WikiText-2 is usually well above that.
#
# Trainer's own eval loop already computes eval_loss -- compute_metrics here
# just derives perplexity from it. eval_pred.predictions holds logits, which
# we don't need for this metric (only the loss matters), so this function is
# intentionally minimal.

def compute_metrics(eval_pred):
    # Trainer calls this AFTER loss is already computed and logged separately
    # as eval_loss -- we recompute perplexity outside, from trainer_output,
    # since eval_pred here only carries logits/labels, not the scalar loss.
    logits, labels = eval_pred
    shift_logits = logits[:, :-1, :]
    shift_labels = labels[:, 1:]
    loss_fn = torch.nn.CrossEntropyLoss()
    loss = loss_fn(
        torch.tensor(shift_logits).reshape(-1, shift_logits.shape[-1]),
        torch.tensor(shift_labels).reshape(-1),
    )
    return {"perplexity": math.exp(loss.item())}

print("\n-- compute_metrics defined --")
dummy_loss = 3.4
print(f"  Dummy loss -> perplexity : {math.exp(dummy_loss):.2f}")
#   29.96


-- compute_metrics defined --
  Dummy loss -> perplexity : 29.96


In [16]:
# ! uv pip install optuna
! uv pip install -q optuna


In [15]:
# ── STEP 7 : TrainingArguments ────────────────────────────────────────────────
# save_strategy="steps" + save_steps=200 (instead of "epoch") is deliberate --
# it gives us multiple checkpoints WITHIN epoch 1, so Step 10 (resume) has
# something concrete to resume from without waiting for a full epoch.

args = TrainingArguments(
    output_dir="./gpt2-wikitext2",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=True,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,          # lower loss (-> lower perplexity) is better
    logging_steps=50,
    save_total_limit=3,
    seed=42,
    report_to="none",
)

print("\n-- TrainingArguments created --")
steps_per_epoch = len(lm_dataset["train"]) // args.per_device_train_batch_size
total_steps = steps_per_epoch * args.num_train_epochs
print(f"  Steps per epoch : {steps_per_epoch}")
print(f"  Total steps     : {total_steps}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.



-- TrainingArguments created --
  Steps per epoch : 2333
  Total steps     : 4666


In [17]:
# ── STEP 8 : Hyperparameter search (Optuna) ───────────────────────────────────
# Requires optuna -- not bundled with transformers, installed below.
#
# hyperparameter_search reuses model_init to spin up a FRESH model per trial.
# Each trial runs a full mini train+eval cycle under `args` with the sampled
# hyperparameters substituted in. n_trials=3 kept low here deliberately --
# this is a demonstration of the mechanism, not a real search budget. A real
# search on this dataset would want 15-30+ trials and a smaller max_steps
# per trial so the search itself doesn't take longer than final training.

import subprocess
import sys

try:
    import optuna
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "optuna"])
    import optuna

search_args = TrainingArguments(
    output_dir="./gpt2-hp-search",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="no",              # don't checkpoint during search -- wastes disk
    logging_steps=100,
    max_steps=200,                    # cap each trial short on purpose
    report_to="none",
)

search_trainer = Trainer(
    model_init=model_init,
    args=search_args,
    train_dataset=lm_dataset["train"],
    eval_dataset=lm_dataset["validation"],
    data_collator=collator,
)

best_run = search_trainer.hyperparameter_search(
    direction="minimize",             # minimizing eval_loss
    backend="optuna",
    n_trials=3,
    hp_space=lambda trial: {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 1e-4, log=True),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [4, 8]),
        "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.1),
    },
)

print("\n-- Hyperparameter search complete --")
print(f"  Best run id           : {best_run.run_id}") # type: ignore
print(f"  Best objective (loss) : {best_run.objective:.4f}") # type: ignore
print(f"  Best hyperparameters  : {best_run.hyperparameters}") # type: ignore
# Apply the winning values to the REAL TrainingArguments before final training:
args.learning_rate = best_run.hyperparameters.get("learning_rate", args.learning_rate) # type: ignore
args.weight_decay = best_run.hyperparameters.get("weight_decay", args.weight_decay) # type: ignore
args.per_device_train_batch_size = best_run.hyperparameters.get( # type: ignore
    "per_device_train_batch_size", args.per_device_train_batch_size
)
print(f"  Applied to args: lr={args.learning_rate}, wd={args.weight_decay}, batch={args.per_device_train_batch_size}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[I 2026-09-11 05:02:45,310] A new study created in memory with name: no-name-4d6db728-6663-40f4-902f-166cf254c6d7


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
100,3.757511,3.530539
200,3.618905,3.498300


[I 2026-09-11 05:04:42,360] Trial 0 finished with value: 3.4982995986938477 and parameters: {'learning_rate': 9.810319568068981e-05, 'per_device_train_batch_size': 8, 'weight_decay': 0.08582877580842807}. Best is trial 0 with value: 3.4982995986938477.


... (3 outputs left out)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss
100,3.937825,3.644073
200,3.784322,3.605125


[I 2026-09-11 05:08:15,454] Trial 2 finished with value: 3.6051251888275146 and parameters: {'learning_rate': 1.53367012508604e-05, 'per_device_train_batch_size': 8, 'weight_decay': 0.09836235204714097}. Best is trial 0 with value: 3.4982995986938477.



-- Hyperparameter search complete --
  Best run id           : 0
  Best objective (loss) : 3.4983
  Best hyperparameters  : {'learning_rate': 9.810319568068981e-05, 'per_device_train_batch_size': 8, 'weight_decay': 0.08582877580842807}
  Applied to args: lr=9.810319568068981e-05, wd=0.08582877580842807, batch=8


In [19]:
# Accelerator state is a singleton -- the search_trainer above locked it to
# search_args' mixed_precision setting (fp16=False, since search_args never
# set fp16). Step 9's Trainer wants fp16=True. Without resetting, transformers
# raises ValueError instead of silently reusing the wrong precision.
from accelerate.state import AcceleratorState
AcceleratorState._reset_state()

In [20]:
# ── STEP 9 : Trainer (final run, with searched hyperparameters) ──────────────
trainer = Trainer(
    model_init=model_init,
    args=args,
    train_dataset=lm_dataset["train"],
    eval_dataset=lm_dataset["validation"],
    data_collator=collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.01)],
)

print("\n-- Trainer created --")
print(f"  Train batches : {len(trainer.get_train_dataloader())}")
print(f"  Eval batches  : {len(trainer.get_eval_dataloader())}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


-- Trainer created --
  Train batches : 2334
  Eval batches  : 121


In [21]:
# ── STEP 10 : Train, then demonstrate resume_from_checkpoint ─────────────────
# Real workflow this simulates: a Colab session disconnects mid-training,
# you reconnect, and need to continue from the last saved checkpoint instead
# of restarting from scratch and losing GPU time.
#
# trainer.train() below runs to completion normally. The resume demonstration
# that follows constructs a NEW Trainer instance and points it at a checkpoint
# folder saved during this run -- exactly what you'd do after a real restart.

print("\n-- Starting training --")
train_result = trainer.train()

print("\n-- Training complete --")
print(f"  Total training time : {train_result.metrics['train_runtime']:.1f}s")
print(f"  Train samples/sec   : {train_result.metrics['train_samples_per_second']:.2f}")
print(f"  Final train loss    : {train_result.metrics['train_loss']:.4f}")

# ── Resume demonstration ──────────────────────────────────────────────────
import os

output_dir = args.output_dir
assert output_dir is not None, "output_dir must be set on TrainingArguments"

checkpoints = sorted(
    [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")],
    key=lambda x: int(x.split("-")[1]),
)
print(f"\n  Checkpoints on disk : {checkpoints}")

if len(checkpoints) >= 2:
    resume_path = os.path.join(output_dir, checkpoints[-2])   # simulate resuming from an earlier one
    print(f"  Simulating restart -- resuming from : {resume_path}")

    resumed_trainer = Trainer(
        model_init=model_init,
        args=args,
        train_dataset=lm_dataset["train"],
        eval_dataset=lm_dataset["validation"],
        data_collator=collator,
    )
    resumed_result = resumed_trainer.train(resume_from_checkpoint=resume_path)
    print(f"  Resumed run final loss : {resumed_result.metrics['train_loss']:.4f}")
else:
    print("  Fewer than 2 checkpoints saved -- lower save_steps to see resume in action")


-- Starting training --


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss
200,3.680642,3.527571
400,3.627395,3.480771
600,3.547387,3.481862
800,3.539613,3.468235
1000,3.471728,3.453496
1200,3.427010,3.438066
1400,3.402064,3.432167
1600,3.414805,3.415810
1800,3.376204,3.411935
2000,3.385877,3.405615


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (26 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


  Resumed run final loss : 1.7866


In [24]:
# ── STEP 11 : Evaluate — perplexity on validation set ─────────────────────────
from typing import cast
from datasets import Dataset
from transformers.utils.notebook import NotebookProgressCallback

print("\n-- Evaluating on validation set --")
trainer.remove_callback(NotebookProgressCallback)   # tracker was reset on_train_end; standalone evaluate() needs it removed
eval_results = trainer.evaluate()

eval_loss = eval_results["eval_loss"]
perplexity = math.exp(eval_loss)

print(f"  eval_loss  : {eval_loss:.4f}")
print(f"  perplexity : {perplexity:.2f}")
# Lower perplexity = better. Compare against the untrained baseline below
# to confirm fine-tuning actually helped:

baseline_model = AutoModelForCausalLM.from_pretrained("gpt2")
baseline_trainer = Trainer(
    model=baseline_model,
    args=args,
    eval_dataset=cast(Dataset, lm_dataset["validation"]),   # required in constructor now — args.eval_strategy="steps" enforces this at init
    data_collator=collator,
)
baseline_trainer.remove_callback(NotebookProgressCallback)

baseline_eval = baseline_trainer.evaluate()
print(f"  Baseline (no fine-tune) perplexity : {math.exp(baseline_eval['eval_loss']):.2f}")
del baseline_model, baseline_trainer


-- Evaluating on validation set --
  eval_loss  : 3.3969
  perplexity : 29.87


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  Baseline (no fine-tune) perplexity : 57.34


In [25]:
# ── STEP 12 : Save model ──────────────────────────────────────────────────────
trainer.save_model("./gpt2-wikitext2-final")
tokenizer.save_pretrained("./gpt2-wikitext2-final")

print("\n-- Model saved --")
print("  Saved to: ./gpt2-wikitext2-final/")
print("  Files   : config.json, model.safetensors, vocab.json, merges.txt, tokenizer_config.json")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


-- Model saved --
  Saved to: ./gpt2-wikitext2-final/
  Files   : config.json, model.safetensors, vocab.json, merges.txt, tokenizer_config.json


In [26]:
print("Part 5: Generation — Full Decoding Strategy Suite — Loads the fine-tuned checkpoint and runs every decoding strategy (greedy, beam, sampling, top-k, top-p, contrastive) plus streaming and batch generation 🚀")

Part 5: Generation — Full Decoding Strategy Suite — Loads the fine-tuned checkpoint and runs every decoding strategy (greedy, beam, sampling, top-k, top-p, contrastive) plus streaming and batch generation 🚀


In [27]:
# ── STEP 13 : Load fine-tuned model for generation ────────────────────────────
gen_tokenizer = AutoTokenizer.from_pretrained("./gpt2-wikitext2-final")
gen_model = AutoModelForCausalLM.from_pretrained("./gpt2-wikitext2-final")
gen_model.eval()

device = next(gen_model.parameters()).device
print(f"  Model device : {device}")

prompt = "The history of artificial intelligence began"
inputs = gen_tokenizer(prompt, return_tensors="pt").to(device)
print(f"  Prompt : {prompt!r}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  Model device : cpu
  Prompt : 'The history of artificial intelligence began'


In [28]:
# ── STEP 13 : Load fine-tuned model for generation ────────────────────────────
gen_tokenizer = AutoTokenizer.from_pretrained("./gpt2-wikitext2-final")
gen_model = AutoModelForCausalLM.from_pretrained("./gpt2-wikitext2-final")

device = "cuda" if torch.cuda.is_available() else "cpu"
gen_model = gen_model.to(device)
gen_model.eval()

print(f"  Model device : {device}")

prompt = "The history of artificial intelligence began"
inputs = gen_tokenizer(prompt, return_tensors="pt").to(device)
print(f"  Prompt : {prompt!r}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  Model device : cuda
  Prompt : 'The history of artificial intelligence began'


In [29]:
# ── STEP 14 : Greedy decoding ──────────────────────────────────────────────────
# Always picks the single most likely next token. Deterministic, fast,
# often repetitive -- watch for the model looping on the same phrase.
output_ids = gen_model.generate( #type:ignore
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    num_beams=1,
)
text = gen_tokenizer.decode(output_ids[0], skip_special_tokens=True)
print("\n-- Greedy --")
print(f"  {text}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



-- Greedy --
  The history of artificial intelligence began with the invention of the computer in the late 1940s . The first artificial intelligence was developed in the 1950s by the University of California at Berkeley , who developed the first artificial intelligence in the 1960s . The first artificial intelligence was developed in the 1970


In [ ]:
# ── STEP 13 : Load fine-tuned model for generation ────────────────────────────
from transformers import GPT2LMHeadModel
from typing import cast

gen_tokenizer = AutoTokenizer.from_pretrained("./gpt2-wikitext2-final")

device = "cuda" if torch.cuda.is_available() else "cpu"

_loaded = AutoModelForCausalLM.from_pretrained("./gpt2-wikitext2-final")
_loaded = _loaded.to(device) #type:ignore
_loaded.eval()

gen_model = cast(GPT2LMHeadModel, _loaded)

print(f"  Model device : {device}")

prompt = "The history of artificial intelligence began"
inputs = gen_tokenizer(prompt, return_tensors="pt").to(device)
print(f"  Prompt : {prompt!r}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  Model device : cuda
  Prompt : 'The history of artificial intelligence began'


In [32]:
# ── STEP 15 : Beam search ───────────────────────────────────────────────────────
# Tracks 4 candidate sequences in parallel, returns the highest-scoring one.
# no_repeat_ngram_size=3 blocks any 3-gram from repeating -- beam search
# without this tends to loop badly on short prompts.
output_ids = gen_model.generate( #type:ignore
    **inputs,
    max_new_tokens=50,
    num_beams=4,
    no_repeat_ngram_size=3,
    early_stopping=True,
)
text = gen_tokenizer.decode(output_ids[0], skip_special_tokens=True)
print("\n-- Beam search (num_beams=4) --")
print(f"  {text}")

# multiple candidates from the same beam search
output_ids = gen_model.generate( #type:ignore
    **inputs,
    max_new_tokens=50,
    num_beams=5,
    num_return_sequences=3,
    early_stopping=True,
    no_repeat_ngram_size=3,
)
print("\n-- Beam search, 3 returned candidates --")
for i, ids in enumerate(output_ids):
    print(f"  [{i}] {gen_tokenizer.decode(ids, skip_special_tokens=True)}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



-- Beam search (num_beams=4) --
  The history of artificial intelligence began in the early 1990s , when a group of researchers from the University of California , Berkeley ( UC Berkeley ) published a paper in the Proceedings of the National Academy of Sciences (PNAS ) on the neural network of the human brain . The paper was

-- Beam search, 3 returned candidates --
  [0] The history of artificial intelligence began in the early 1990s , when a group of researchers at the University of California , Berkeley ( UC Berkeley ) published a paper in the Proceedings of the National Academy of Sciences (PNAS ) that showed that artificial intelligence could be used to solve problems .
  [1] The history of artificial intelligence began in the early 1990s , when a group of researchers at the University of California , Berkeley ( UC Berkeley ) published a paper in the Proceedings of the National Academy of Sciences (PNAS ) that showed that artificial intelligence could be used to solve problems in

In [33]:
# ── STEP 16 : Sampling ──────────────────────────────────────────────────────────
# Randomly samples from the full probability distribution at each step.
# temperature=1.0 leaves the distribution unchanged; setting the same seed
# with do_sample=True still varies run to run unless torch.manual_seed is fixed.
torch.manual_seed(42)
output_ids = gen_model.generate( #type :ignore
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=1.0,
)
text = gen_tokenizer.decode(output_ids[0], skip_special_tokens=True)
print("\n-- Sampling (temperature=1.0) --")
print(f"  {text}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



-- Sampling (temperature=1.0) --
  The history of artificial intelligence began in the 1940s when researchers in Britain and Australia suggested that some individuals may have been driven away by AI technology when they faced serious difficulties . Researchers who knew little about such concepts also argued that AI was a real problem and called it a " problem where


In [34]:
# ── STEP 17 : Top-k sampling ────────────────────────────────────────────────────
torch.manual_seed(42)
output_ids = gen_model.generate( #type :ignore
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    top_k=50,
    temperature=0.8,
)
text = gen_tokenizer.decode(output_ids[0], skip_special_tokens=True)
print("\n-- Top-k sampling (k=50, temp=0.8) --")
print(f"  {text}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



-- Top-k sampling (k=50, temp=0.8) --
  The history of artificial intelligence began in the 1940s when researchers in Britain and Australia suggested that some types of intelligent machines could be created with the help of intelligent machines . However , this was not always the case . The first successful attempt to attempt this was conducted in the 1970s by


In [35]:
# ── STEP 18 : Top-p (nucleus) sampling ─────────────────────────────────────────
torch.manual_seed(42)
output_ids = gen_model.generate( #type :ignore
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    top_p=0.92,
    temperature=0.8,
)
text = gen_tokenizer.decode(output_ids[0], skip_special_tokens=True)
print("\n-- Top-p sampling (p=0.92, temp=0.8) --")
print(f"  {text}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



-- Top-p sampling (p=0.92, temp=0.8) --
  The history of artificial intelligence began in the 1940s when researchers in the field of neural net theory developed the idea of artificial intelligence , which was to make intelligent machines capable of reasoning and reasoning . The idea was first described by physicist John Watson in the 1960s , and the idea was


In [36]:
# ── STEP 19 : Combined top-k + top-p + repetition penalty ─────────────────────
# This combination is the de facto default for open-ended creative generation --
# top_k prunes the long tail, top_p adapts candidate count per step,
# repetition_penalty discourages loops that top_k/top_p alone don't fully stop.
torch.manual_seed(42)
output_ids = gen_model.generate( #type :ignore
    **inputs,
    max_new_tokens=80,
    do_sample=True,
    temperature=0.7,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)
text = gen_tokenizer.decode(output_ids[0], skip_special_tokens=True)
print("\n-- Combined top-k + top-p + repetition_penalty --")
print(f"  {text}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



-- Combined top-k + top-p + repetition_penalty --
  The history of artificial intelligence began in the 1940s , when the Japanese government had begun to develop a system called AI . The idea was that it would be possible for intelligent machines and humans alike could be made to understand each other 's thoughts , feelings or desires , while simultaneously possessing knowledge about their environment and reality itself ( i.e .. how they lived , where they traveled ) ; by this time the only way to make them


In [38]:
# ── STEP 20 : Contrastive search ────────────────────────────────────────────────
# Deterministic (no random sampling) but coherence-aware -- penalizes
# candidates too similar to already-generated context. Good middle ground
# between beam search's blandness and sampling's variance.
#
# Newer transformers versions moved contrastive search out of core into a
# community `custom_generate` repo (transformers-community/contrastive-search).
# trust_remote_code=True is required because generate() now downloads and
# executes that repo's decoding code at call time.
output_ids = gen_model.generate(
    **inputs,
    max_new_tokens=80,
    penalty_alpha=0.6,
    top_k=4,
    trust_remote_code=True,
)
text = gen_tokenizer.decode(output_ids[0], skip_special_tokens=True)
print("\n-- Contrastive search (penalty_alpha=0.6, top_k=4) --")
print(f"  {text}")

generate.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/transformers-community/contrastive-search:
- custom_generate/generate.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Passing `generation_config` together with generation-related arguments=({'top_k', 'cache_implementation', 'max_new_tokens', 'penalty_alpha'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
An assistant model is provided, using a dynamic cache instead of a cache of type='dynamic_full'.



-- Contrastive search (penalty_alpha=0.6, top_k=4) --
  The history of artificial intelligence began in a book by the British mathematician and linguist William Shakespeare . In his 1819 book The Language of the Mind , he described the " artificial intelligence " as " an artificial intelligence that could be made to do anything " , and he described it as " a vast and complex machine " . 
 The term " artificial intelligence " was coined by the American sociologist and linguist Richard Feynman


In [39]:
# ── STEP 21 : Streaming — TextStreamer (prints as it generates) ──────────────
streamer = TextStreamer(gen_tokenizer, skip_prompt=True, skip_special_tokens=True)
print("\n-- TextStreamer output --")
_ = gen_model.generate(**inputs, max_new_tokens=50, streamer=streamer, do_sample=False)  # type: ignore

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



-- TextStreamer output --
 with the invention of the computer in the late 1940s . The first artificial intelligence was developed in the 1950s by the University of California at Berkeley , who developed the first artificial intelligence in the 1960s . The first artificial intelligence was developed in the 1970


In [41]:
# ── STEP 22 : Streaming — TextIteratorStreamer (for serving code) ─────────────
iter_streamer = TextIteratorStreamer(gen_tokenizer, skip_prompt=True, skip_special_tokens=True)

generation_kwargs = dict(
    **inputs,
    max_new_tokens=50,
    streamer=iter_streamer,
    do_sample=False,
)
thread = threading.Thread(target=gen_model.generate, kwargs=generation_kwargs)  # type: ignore
thread.start()

print("\n── TextIteratorStreamer output (token by token) ──")
generated_text = ""
for token in iter_streamer:
    generated_text += token
    print(token, end="", flush=True)
thread.join()
print(f"\n\n  Full streamed text : {generated_text}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



── TextIteratorStreamer output (token by token) ──
 with the invention of the computer in the late 1940s . The first artificial intelligence was developed in the 1950s by the University of California at Berkeley , who developed the first artificial intelligence in the 1960s . The first artificial intelligence was developed in the 1970

  Full streamed text :  with the invention of the computer in the late 1940s . The first artificial intelligence was developed in the 1950s by the University of California at Berkeley , who developed the first artificial intelligence in the 1960s . The first artificial intelligence was developed in the 1970


In [43]:
# ── STEP 23 : Batch generation ──────────────────────────────────────────────────
gen_tokenizer.padding_side = "left"

batch_prompts = [
    "Tanvi studied machine learning because",
    "The transformer architecture changed NLP by",
    "In 2017, researchers introduced",
]
batch_inputs = gen_tokenizer(batch_prompts, padding=True, return_tensors="pt").to(device)

output_ids = gen_model.generate(  # type: ignore
    **batch_inputs,
    max_new_tokens=40,
    do_sample=False,
    pad_token_id=gen_tokenizer.pad_token_id, #type:ignore
)

print("\n── Batch generation ──")
for i, ids in enumerate(output_ids):
    print(f"  [{i}] {gen_tokenizer.decode(ids, skip_special_tokens=True)}")


── Batch generation ──
  [0] Tanvi studied machine learning because he believed that it was possible to learn from a single source . He also believed that the most important thing to learn from a single source was to learn from the source . He believed that the most important
  [1] The transformer architecture changed NLP by adding a large number of small , low @-@ power components . The first phase of the project was to build a large @-@ scale power plant , which would be located in the city of
  [2] In 2017, researchers introduced a new method of measuring the amount of water in a sample of the water sample . The researchers used a technique called " water @-@ capture " , which allows the researchers to measure the amount of


In [ ]:
# ── DRY RUN : WikiText-2 train set, 1 GPU (T4), first 3 batches ─────────────
#
# Initial state:
#   model   = GPT-2 (124M) loaded fresh via model_init()
#   dataset = ~15,000 blocks of 128 tokens each (post grouping)
#   LR      = 0.0 (warmup starts at 0)
#
# state -> action -> new state
#
# Batch 1 (global_step=1):
#   state   : LR=0.0, loss=undefined
#   action  : collator stacks 8 pre-chunked blocks -> input_ids [8, 128]
#             labels = input_ids shifted internally by the model
#             forward -> logits [8, 128, 50257], loss ~ ln(50257) ~ 10.8
#               (random-ish loss for an UNTRAINED head would be this high;
#                GPT-2 is already pretrained, so actual starting loss on
#                in-domain text is much lower, commonly 3-4)
#             backward -> gradients for 124M params
#             optimizer step -> LR ramps per warmup schedule
#   new state: global_step=1, loss dropping from pretrained baseline
#
# Batch 200 (first checkpoint + eval, save_steps=200):
#   state   : LR near peak (5e-5, post hyperparameter search value)
#   action  : Trainer pauses training, runs full eval pass on validation blocks
#             compute_metrics -> perplexity computed from eval_loss
#             checkpoint saved -> ./gpt2-wikitext2/checkpoint-200/
#             EarlyStoppingCallback compares against best_metric so far
#   new state: checkpoint-200 on disk, training resumes
#
# Simulated restart after checkpoint-400 exists:
#   action  : new Trainer() instance built from model_init (fresh, empty
#             optimizer state) then trainer.train(resume_from_checkpoint=...)
#             restores: model weights, optimizer state (AdamW momentum/variance
#             buffers), LR scheduler position, global_step counter
#   new state: training continues from step 400's exact LR and step count --
#              NOT from step 0, and NOT from step 400 with a reset optimizer
#
# End of training:
#   load_best_model_at_end=True -> loads whichever checkpoint had lowest eval_loss
#   trainer.evaluate() -> confirms final perplexity
#   Generation section loads this saved checkpoint fresh from disk, independent
#   of whatever's left in GPU memory from training -- confirms the save/load
#   round-trip actually works, not just that training ran.